# Practice — KNN on `mpg.csv`

Same six stages as the Titanic notebook, on your own.

**Question:** given a car's specifications, was it built in the USA, Europe or Japan?

Target: `origin` &nbsp;·&nbsp; File: `mpg.csv`

In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
df = pd.read_csv('/Users/alokkumardalei/Desktop/ml-assignment/mpg.csv')
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,usa,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite
3,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst
4,17.0,8,302.0,140.0,3449,10.5,70,usa,ford torino


---
## 1. Look at the data

> **Flow:** Shape, columns, what is missing.

Run `.info()` and `.shape`. Which column has missing values, and how many?

In [2]:

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     398 non-null    int64  
 2   displacement  398 non-null    float64
 3   horsepower    392 non-null    float64
 4   weight        398 non-null    int64  
 5   acceleration  398 non-null    float64
 6   model_year    398 non-null    int64  
 7   origin        398 non-null    str    
 8   name          398 non-null    str    
dtypes: float64(4), int64(3), str(2)
memory usage: 28.1 KB


How many cars from each `origin`? Use `value_counts()`.

In [3]:

df['origin'].value_counts()

origin
usa       249
japan      79
europe     70
Name: count, dtype: int64

---
## 2. Stage 1 — Data Cleaning

> **Flow:** Fix missing values, drop unusable columns.

Two jobs:

1. `horsepower` has blanks. Fill them with the median.
2. Drop `name`. In one line below, say why it cannot help the model.

In [12]:
df.fillna(df['horsepower'].median(),inplace=True)
df=df.drop(columns=['name'])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     398 non-null    int64  
 2   displacement  398 non-null    float64
 3   horsepower    398 non-null    float64
 4   weight        398 non-null    int64  
 5   acceleration  398 non-null    float64
 6   model_year    398 non-null    int64  
 7   origin        398 non-null    str    
dtypes: float64(4), int64(3), str(1)
memory usage: 25.0 KB


*Why `name` cannot help:*

---
## 3. Features and Target

> **Flow:** `X` is everything the model looks at. `y` is the answer.

In [14]:
from pandas._libs import indexing

x=df.drop(columns=['origin'])
print(x)
y=df['origin']
y.head()

      mpg  cylinders  displacement  horsepower  weight  acceleration  \
0    18.0          8         307.0       130.0    3504          12.0   
1    15.0          8         350.0       165.0    3693          11.5   
2    18.0          8         318.0       150.0    3436          11.0   
3    16.0          8         304.0       150.0    3433          12.0   
4    17.0          8         302.0       140.0    3449          10.5   
..    ...        ...           ...         ...     ...           ...   
393  27.0          4         140.0        86.0    2790          15.6   
394  44.0          4          97.0        52.0    2130          24.6   
395  32.0          4         135.0        84.0    2295          11.6   
396  28.0          4         120.0        79.0    2625          18.6   
397  31.0          4         119.0        82.0    2720          19.4   

     model_year  
0            70  
1            70  
2            70  
3            70  
4            70  
..          ...  
393      

0    usa
1    usa
2    usa
3    usa
4    usa
Name: origin, dtype: str

---
## 4. Stage 2 — Train/Test Split

> **Flow:** Hide some rows before preparing anything.

Use `test_size=0.2`, `random_state=0`, `stratify=y`.

In [23]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test= train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

---
## 5. Stage 3 — Feature Engineering

> **Flow:** Put every column on the same scale.

**No encoding needed here.** After dropping `name`, every feature is already a number,
so there is no text column left for `OneHotEncoder`. That happens in real projects too.

Print the min and max of each feature. Which column has the largest range?

In [17]:
a=df.min()
b=df.max()
print(a)


mpg                9.0
cylinders            3
displacement      68.0
horsepower        46.0
weight            1613
acceleration       8.0
model_year          70
origin          europe
dtype: object


Now scale. `StandardScaler` — `fit_transform` on train, `transform` on test.

In [24]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test);


---
## 6. Stages 4 and 5 — Train and Predict

> **Flow:** `.fit()` learns, `.predict()` answers.

**First without scaling.** Train `KNeighborsClassifier()` on the unscaled data and print the accuracy.

In [28]:
scaled_model=KNeighborsClassifier()
scaled_model.fit(x_train_scaled,y_train)
y_pre=scaled_model.predict(x_test_scaled)
print(accuracy_score(y_pre,y_test))


0.7


**Now with scaling.** Same model, same `k`, scaled data.

In [27]:
model=KNeighborsClassifier()
model.fit(x_train,y_train)
y_pre=model.predict(x_test)
print(accuracy_score(y_pre,y_test))


0.6875


---
## 7. Stage 6 — Compare

> **Flow:** Two numbers, one difference.

Print both accuracies together. Which is higher, and by how much?

*What changed between the two runs:*

---
## 8. Choosing k

> **Flow:** `k` is a dial. Try a few settings and look.

Run a loop over `k = 1, 3, 5, 7, 9, 11, 15, 21` on the **scaled** data.
Print `k` and its accuracy on each line.

In [30]:
for k in [1,3,5,7,9,11,15,21]:
    m=KNeighborsClassifier(n_neighbors=k)
    m.fit(x_train_scaled,y_train)
    y_pre=m.predict(x_test_scaled)
    print(accuracy_score(y_pre,y_test))


0.7375
0.7125
0.7
0.675
0.7125
0.725
0.7625
0.725


*Best k:* &nbsp;&nbsp; *Its accuracy:*

Does the accuracy change a lot across k, or stay roughly flat?

*Your answers:*